In [1]:
import warnings
warnings.filterwarnings("ignore")

# Quick environment check
import importlib, sys

required = ["torch", "transformers", "datasets", "peft", "sklearn", "matplotlib", "flask", "requests"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg.replace("-", "_")) is None]

if missing:
    print(f"\u274c Missing packages: {missing}")
    print("   Run: pip install torch transformers datasets peft accelerate scikit-learn matplotlib flask requests")
else:
    print("\u2705 All packages available")
    import torch
    print(f"   PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}")


import os
import textwrap
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore")



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

✅ All packages available
   PyTorch 2.10.0+cpu  |  CUDA: False | MPS: False


In [2]:
target_model_name = 'tinyllama:latest'

In [3]:
import requests, json, textwrap

# Create a session that ignores proxy env vars (bypasses VPN for local Ollama)
SESSION = requests.Session()
SESSION.trust_env = False  # equivalent of ollama.Client(..., trust_env=False)


def ollama_list_models():
    r = SESSION.get("http://localhost:11434/api/tags", timeout=30)
    r.raise_for_status()
    data = r.json()
    return [m.get("name", "") for m in (data.get("models") or [])]

def check_ollama_ready():
    models = ollama_list_models()
    print(f"Ollama reachable. {len(models)} model(s) found.")
    if target_model_name not in models:
        print("Warning: chat model tinyllama not found in `ollama list`.")

def ask_ollama(prompt, model=target_model_name, system=None, temperature=0.7):
    """Send a prompt to Ollama and return the response text."""
    url = "http://localhost:11434/api/chat"
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": 256},
    }

    try:
        resp = SESSION.post(url, json=payload, timeout=120)
        resp.raise_for_status()
        data = resp.json()

        # /api/chat returns message.content; keep /api/generate fallback for robustness.
        content = (data.get("message") or {}).get("content")
        if content:
            return content
        if "response" in data:
            return data["response"]
        return f"⚠️ Unexpected Ollama response format: {data}"
    except requests.ConnectionError:
        return "⚠️ Ollama not running. Start it with: ollama serve"
    except Exception as e:
        return f"⚠️ Error: {e}"

def show(label, text, width=90):
    """Pretty-print a response."""
    sep = "=" * width
    print(f"\n{sep}")
    print(f"  {label}")
    print(sep)
    pretty_print(text)
    print()


check_ollama_ready()

Ollama reachable. 6 model(s) found.


In [4]:
# ── A question that demands domain expertise ─────────────────
medical_q = (
    "A 62-year-old male presents with crushing substernal chest pain "
    "radiating to the left arm, diaphoresis, and ST-elevation in leads II, III, and aVF. "
    "What is the most likely diagnosis and immediate management?"
)

response = ask_ollama(medical_q, model=target_model_name)
show("TinyLlama (Base) \u2014 Medical Question", response)


  TinyLlama (Base) — Medical Question
The most likely diagnosis is aortic stenosis, and immediate management would be
to perform an echocardiogram to rule out other causes for the heart murmur, such
as valvular heart disease or myocardial infarction. Aortic stenosis can cause
severe symptoms like cruching chest pain, and if the diagnosis is confirmed, the
treatment options would depend on the severity of the disease and the patient's
overall health. The patient should be monitored closely for complications and
given medication to manage symptoms.



In [5]:
# ── Customer support: we want concise, empathetic, branded ──
support_q = (
    "Customer says: I have been on hold for 45 minutes and my order "
    "still has not shipped. This is ridiculous.\n\n"
    "Write a response as a support agent for TechMart Electronics."
)

response = ask_ollama(support_q, model=target_model_name)
show("TinyLlama (Base) \u2014 Customer Support", response)


  TinyLlama (Base) — Customer Support
Dear Customer,  I am sorry to hear about your frustrating experience with
TechMart Electronics. I understand how inconvenient it can be to wait for your
order to be shipped after placing it, especially for such a long time.  I have
reached out to our customer service team to inquire about your order status and
ask if they have any updates. Please let me know if there are any updates or
concerns regarding your order.  We take customer satisfaction very seriously and
have a team dedicated to resolving any issues that may arise during the order
process. If you have not received any updates, please let us know and we will do
our best to resolve the issue for you.  Thank you for your patience and
understanding during this process. We value your business and will do everything
in our power to ensure that your order is shipped promptly and safely.  Best
regards,  [Your Name]  [TechMart Electronics]



In [6]:
# ── Structured output: base models often fumble formats ─────
json_q = (
    "Extract the following from this text and return ONLY valid JSON:\n\n"
    "Text: Dr. Sarah Chen, a cardiologist at Mayo Clinic, published a study on "
    "atrial fibrillation treatment using novel anticoagulants in March 2024.\n\n"
    "Required JSON keys: name, specialty, institution, topic, date. Also I want only and only valid JSON with no extra commentary or text. If any info is missing, use null for that key. AGAIN ONLY VALID JSON WITH NO EXTRA TEXT OR COMMENTARY."
)

response = ask_ollama(json_q, model=target_model_name)
show("TinyLlama (Base) \u2014 Structured Extraction", response)

# Let's see if it's actually valid JSON

try:
    extracted = json.loads(response)
    print("\u2705 Valid JSON extracted:")
    print(json.dumps(extracted, indent=2))
except json.JSONDecodeError as e:
    print(f"\u274c Invalid JSON: {e}")


  TinyLlama (Base) — Structured Extraction
{   "name": "Dr. Sarah Chen",   "specialty": "Cardiologist",   "institution":
"Mayo Clinic",   "topic": "Atrial Fibrillation Treatment Using Novel AntiCoagu
Lamnts",   "date": "2024-03-01",   "validJSON": {     "name": "Dr. Sarah Chen",
"specialty": "Cardiologist",     "institution": "Mayo Clinic",     "topic":
"Atrial Fibrillation Treatment Using Novel AntiCoagu Lamnts",     "date":
"2024-03-01",     "validJSON": true   } }

✅ Valid JSON extracted:
{
  "name": "Dr. Sarah Chen",
  "specialty": "Cardiologist",
  "institution": "Mayo Clinic",
  "topic": "Atrial Fibrillation Treatment Using Novel AntiCoagu Lamnts",
  "date": "2024-03-01",
  "validJSON": {
    "name": "Dr. Sarah Chen",
    "specialty": "Cardiologist",
    "institution": "Mayo Clinic",
    "topic": "Atrial Fibrillation Treatment Using Novel AntiCoagu Lamnts",
    "date": "2024-03-01",
    "validJSON": true
  }
}


LoRA

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = r"C:\Users\arunk\OneDrive\Documents\GenAI\LLM\distilgpt2"


print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Count parameters
total_params = sum(p.numel() for p in base_model.parameters())
print(f"\u2705 Loaded: {total_params:,} parameters ({total_params/1e6:.1f}M)")

Loading C:\Users\arunk\OneDrive\Documents\GenAI\LLM\distilgpt2...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6284.47it/s]

✅ Loaded: 81,912,576 parameters (81.9M)


In [11]:
# ── Step 2: Test the base model BEFORE fine-tuning ───────────
def generate_text(model, prompt, max_new_tokens=100):
    """Generate text from a model given a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

# ── Test with a support-style prompt ─────────────────────────
test_prompt = "Customer: My order #4521 hasn't arrived after 2 weeks.\nAgent:"

print("\U0001f4cb BEFORE Fine-Tuning:")
print(f"   Prompt: {test_prompt}")
print(f"   Response: {generate_text(base_model, test_prompt)}")

📋 BEFORE Fine-Tuning:
   Prompt: Customer: My order #4521 hasn't arrived after 2 weeks.
Agent:
   Response: Hi, so I know you were in a hurry to get your orders out early for the holidays and we haven‡t had many cancellations of my items since this past weekend - it makes me feel like something was missing."


In [12]:
# -- Step 3: Build customer support QA-style pairs --
support_examples = [
    (
        "Customer: My order hasn't arrived yet.\nAgent:",
        " I'm sorry to hear that! Let me look into your order right away. Could you please share your order number so I can track it for you?\n",
    ),
    (
        "Customer: I want a refund for my broken headphones.\nAgent:",
        " I completely understand your frustration. We'll get this sorted out. I'm initiating a refund for you right now.\n",
    ),
    (
        "Customer: How do I reset my password?\nAgent:",
        " Great question! Go to Settings > Account > Reset Password. You'll receive an email with a reset link.\n",
    ),
    (
        "Customer: Your app keeps crashing on my phone.\nAgent:",
        " I'm sorry about that! Let's troubleshoot: first, try updating the app to the latest version. If that doesn't help, clear the app cache.\n",
    ),
    (
        "Customer: I was charged twice for the same item.\nAgent:",
        " Oh no, that shouldn't happen! I can see the duplicate charge. I'm processing a refund for the extra charge right now.\n",
    ),
    (
        "Customer: Can I change my delivery address?\nAgent:",
        " Of course! If your order hasn't shipped yet, I can update the address right away. Could you please provide the new address?\n",
    ),
    (
        "Customer: The product I received is the wrong color.\nAgent:",
        " I apologize for the mix-up! I'll arrange a replacement in the correct color to be shipped out today at no extra cost!\n",
    ),
    (
        "Customer: I need to cancel my subscription.\nAgent:",
        " I understand. I can process that cancellation right now. Your subscription will remain active until the end of your current billing period.\n",
    ),
    (
        "Customer: The website won't accept my coupon code.\nAgent:",
        " Let me check that code for you! Sometimes codes are case-sensitive or have an expiration date. What code are you using?\n",
    ),
    (
        "Customer: I never received my confirmation email.\nAgent:",
        " No worries! Let me resend that for you. Could you confirm the email address on your account?\n",
    ),
    (
        "Customer: My package arrived damaged.\nAgent:",
        " I'm so sorry about that! I'm sending a replacement right away at no charge. You don't need to return the damaged item.\n",
    ),
    (
        "Customer: How long does shipping usually take?\nAgent:",
        " Standard shipping typically takes 5-7 business days. We also offer express (2-3 days) and overnight options at checkout.\n",
    ),
]

In [13]:
# ── Tokenize the dataset ──────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class SupportDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=256):
        self.items = []

        for prompt, answer in pairs:
            full_text = prompt + answer
            enc = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )

            input_ids = enc["input_ids"].squeeze(0)
            attention_mask = enc["attention_mask"].squeeze(0)

            labels = input_ids.clone()

            prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            prompt_len = len(prompt_ids)

            labels[:prompt_len] = -100
            labels[attention_mask == 0] = -100

            self.items.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

dataset = SupportDataset(support_examples, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f"\u2705 Dataset ready: {len(dataset)} examples, batch_size=4")


✅ Dataset ready: 12 examples, batch_size=4


In [14]:
# print layers in the model

for name, layer in base_model.named_modules():
    print(name)


transformer
transformer.wte
transformer.wpe
transformer.drop
transformer.h
transformer.h.0
transformer.h.0.ln_1
transformer.h.0.attn
transformer.h.0.attn.c_attn
transformer.h.0.attn.c_proj
transformer.h.0.attn.attn_dropout
transformer.h.0.attn.resid_dropout
transformer.h.0.ln_2
transformer.h.0.mlp
transformer.h.0.mlp.c_fc
transformer.h.0.mlp.c_proj
transformer.h.0.mlp.act
transformer.h.0.mlp.dropout
transformer.h.1
transformer.h.1.ln_1
transformer.h.1.attn
transformer.h.1.attn.c_attn
transformer.h.1.attn.c_proj
transformer.h.1.attn.attn_dropout
transformer.h.1.attn.resid_dropout
transformer.h.1.ln_2
transformer.h.1.mlp
transformer.h.1.mlp.c_fc
transformer.h.1.mlp.c_proj
transformer.h.1.mlp.act
transformer.h.1.mlp.dropout
transformer.h.2
transformer.h.2.ln_1
transformer.h.2.attn
transformer.h.2.attn.c_attn
transformer.h.2.attn.c_proj
transformer.h.2.attn.attn_dropout
transformer.h.2.attn.resid_dropout
transformer.h.2.ln_2
transformer.h.2.mlp
transformer.h.2.mlp.c_fc
transformer.h.2.mlp

In [15]:
# ── Step 4: Apply LoRA adapters ───────────────────────────────
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # Rank of the low-rank matrices
    lora_alpha=32,          # Scaling factor
    lora_dropout=0.1,       # Regularization
    target_modules=["c_attn", "c_proj"],  # Which layers get LoRA
)

model = get_peft_model(base_model, lora_config) 

# ── The big reveal: parameter comparison ─────────────────────
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("  LoRA Parameter Report")
print("=" * 60)
print(f"  Total parameters:     {all_params:>12,}")
print(f"  Trainable (LoRA):     {trainable_params:>12,}")
print(f"  Frozen (base model):  {all_params - trainable_params:>12,}")
print(f"  Trainable %:          {100 * trainable_params / all_params:>11.4f}%")
print("=" * 60)
print(f"\n\U0001f4a1 We're training only {trainable_params:,} out of {all_params:,} parameters!")
print(f"   That's like editing {trainable_params/all_params*100:.2f}% of a textbook instead of rewriting it.")


  LoRA Parameter Report
  Total parameters:       82,318,080
  Trainable (LoRA):          405,504
  Frozen (base model):    81,912,576
  Trainable %:               0.4926%

💡 We're training only 405,504 out of 82,318,080 parameters!
   That's like editing 0.49% of a textbook instead of rewriting it.


In [16]:
# print layers in the model

for name, layer in model.named_modules():
    print(name)


base_model
base_model.model
base_model.model.transformer
base_model.model.transformer.wte
base_model.model.transformer.wpe
base_model.model.transformer.drop
base_model.model.transformer.h
base_model.model.transformer.h.0
base_model.model.transformer.h.0.ln_1
base_model.model.transformer.h.0.attn
base_model.model.transformer.h.0.attn.c_attn
base_model.model.transformer.h.0.attn.c_attn.base_layer
base_model.model.transformer.h.0.attn.c_attn.lora_dropout
base_model.model.transformer.h.0.attn.c_attn.lora_dropout.default
base_model.model.transformer.h.0.attn.c_attn.lora_A
base_model.model.transformer.h.0.attn.c_attn.lora_A.default
base_model.model.transformer.h.0.attn.c_attn.lora_B
base_model.model.transformer.h.0.attn.c_attn.lora_B.default
base_model.model.transformer.h.0.attn.c_attn.lora_embedding_A
base_model.model.transformer.h.0.attn.c_attn.lora_embedding_B
base_model.model.transformer.h.0.attn.c_attn.lora_magnitude_vector
base_model.model.transformer.h.0.attn.c_proj
base_model.model.

In [17]:
# ── Step 5: Training loop ─────────────────────────────────────
from torch.optim import AdamW
import time

optimizer = AdamW(model.parameters(), lr=5e-4)
model.train()

NUM_EPOCHS = 10
losses = []

print("\U0001f3cb\ufe0f Training LoRA adapters...")
print("-" * 50)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)

    bar = "\u2588" * int(30 * (epoch + 1) / NUM_EPOCHS) + "\u2591" * (30 - int(30 * (epoch + 1) / NUM_EPOCHS))
    print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS} |{bar}| Loss: {avg_loss:.4f}")

elapsed = time.time() - start_time
print(f"\n\u2705 Training complete in {elapsed:.1f}s")


🏋️ Training LoRA adapters...
--------------------------------------------------


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Epoch  1/10 |███░░░░░░░░░░░░░░░░░░░░░░░░░░░| Loss: 3.2736
  Epoch  2/10 |██████░░░░░░░░░░░░░░░░░░░░░░░░| Loss: 3.0972
  Epoch  3/10 |█████████░░░░░░░░░░░░░░░░░░░░░| Loss: 2.8683
  Epoch  4/10 |████████████░░░░░░░░░░░░░░░░░░| Loss: 2.7105
  Epoch  5/10 |███████████████░░░░░░░░░░░░░░░| Loss: 2.4840
  Epoch  6/10 |██████████████████░░░░░░░░░░░░| Loss: 2.3263
  Epoch  7/10 |█████████████████████░░░░░░░░░| Loss: 2.1933
  Epoch  8/10 |████████████████████████░░░░░░| Loss: 1.9707
  Epoch  9/10 |███████████████████████████░░░| Loss: 1.8115
  Epoch 10/10 |██████████████████████████████| Loss: 1.7261

✅ Training complete in 189.9s


In [18]:
# ── Compare before and after ──────────────────────────────────
model.eval()

test_prompts = [
    "Customer: My order #4521 hasn't arrived after 2 weeks.\nAgent:",
    "Customer: I want a refund.\nAgent:",
    "Customer: Your app keeps crashing.\nAgent:",
]

from transformers import AutoModelForCausalLM
base_for_comparison = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("\U0001f52c Side-by-Side Comparison: Base vs LoRA Fine-Tuned")
print("=" * 80)

for prompt in test_prompts:
    print(f"\n\U0001f4cb Prompt: {prompt}")
    print("-" * 80)

    base_response = generate_text(base_for_comparison, prompt, max_new_tokens=80)
    print(f"  \U0001f7e1 BASE:      {base_response[:200]}")

    ft_response = generate_text(model, prompt, max_new_tokens=80)
    print(f"  \U0001f7e2 FINE-TUNED: {ft_response[:200]}")
    print()


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 4421.24it/s]


🔬 Side-by-Side Comparison: Base vs LoRA Fine-Tuned

📋 Prompt: Customer: My order #4521 hasn't arrived after 2 weeks.
Agent:
--------------------------------------------------------------------------------
  🟡 BASE:      I've been ordering from my local store, so there's a lot of work to do before shipping out!
  🟢 FINE-TUNED: I'll update that for you right away! You can receive one today at checkout now and continue receiving the correct number either way on your account or in an e-mail address -- please wait until Monday 


📋 Prompt: Customer: I want a refund.
Agent:
--------------------------------------------------------------------------------
  🟡 BASE:      My job is to help find the best place for me and my family, so if we don't have that same customer service then there's nothing you can do about this situation."
  🟢 FINE-TUNED: Well, that could end today! Let me look into the process right away — can we replace it? Could you please share your order with us at checkout for free